# Saving and Loading Models

## Option 1: `torch.save()`

Save **complete** tensor models. `torch.save()` can use any dictionary for saving.

- serialized and not human-readable
- Disadvantage: Serialized data is bound to specific classes and exact directory structure used when the model is saved.

```
torch.save(model, PATH)

model = torch.load(PATH) # model class defined somewhere
model.eval()
```

## Option 2 (Recommended)

If we just want to save our model and use it later for inference, it is enough to only save the parameters.

- save state dictionary

```
torch.save(model.state_dict(), PATH)

# model must be created again
model = Model(*args, **kwargs)
model.load_state_dict(torch.load(PATH))
model.eval()
```

## In Practice

In [19]:
import torch
import torch.nn as nn

class Model(nn.Module):
    def __init__(self, n_input_features):
        super(Model, self).__init__()
        self.linear = nn.Linear(n_input_features, 1)

    def forward(self, x):
        y_pred = torch.sigmoid(self.linear(x))
        return y_pred

model = Model(n_input_features=6)

# train the model ...

### Option 1

In [20]:
# save the model
FILE = "model.pth"
torch.save(model, FILE)

In [21]:
# load the model

model = torch.load(FILE, weights_only=False)
model.eval() # set to eval method

for param in model.parameters():
    print(param)

Parameter containing:
tensor([[-0.0031,  0.3723, -0.3986, -0.2375,  0.1004, -0.3590]],
       requires_grad=True)
Parameter containing:
tensor([0.1414], requires_grad=True)


In [22]:
print(model.state_dict())

OrderedDict({'linear.weight': tensor([[-0.0031,  0.3723, -0.3986, -0.2375,  0.1004, -0.3590]]), 'linear.bias': tensor([0.1414])})


### Option 2 (Recommended)

In [23]:
# save the model - only weights
FILE = "model2.pth"
torch.save(model.state_dict(), FILE)

In [24]:
# load the model
loaded_model = Model(n_input_features=6)
loaded_model.load_state_dict(torch.load(FILE))
loaded_model.eval()

for param in loaded_model.parameters():
    print(param)

Parameter containing:
tensor([[-0.0031,  0.3723, -0.3986, -0.2375,  0.1004, -0.3590]],
       requires_grad=True)
Parameter containing:
tensor([0.1414], requires_grad=True)


## Common Way of Saving a Checkpoint during Training

If we have an optimizer, we can also see the state dictionary of the optimizer.

In [25]:
learning_rate = 0.01
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

print(optimizer.state_dict())

{'state': {}, 'param_groups': [{'lr': 0.01, 'momentum': 0, 'dampening': 0, 'weight_decay': 0, 'nesterov': False, 'maximize': False, 'foreach': None, 'differentiable': False, 'fused': None, 'params': [0, 1]}]}


During training, if we want to stop at a certain point and save a checkpoint:

In [27]:
# save checkpoint
checkpoint = {
    "epoch": 90,
    "model_state": model.state_dict(),
    "optim_state": optimizer.state_dict()
}

torch.save(checkpoint, "checkpoint1.pth")

In [30]:
# load checkpoint
loaded_checkpoint = torch.load("checkpoint1.pth")

epoch = loaded_checkpoint["epoch"]

# create the model and optimizer again
model = Model(n_input_features=6)
optimizer = torch.optim.SGD(model.parameters(), lr=0) # correct learning rate will be loaded

model.load_state_dict(checkpoint["model_state"])
optimizer.load_state_dict(checkpoint["optim_state"])

print(f"Model state dict: {model.state_dict()}")
print(f"Optimizer state dict: {optimizer.state_dict()}")

# continue training ...

Model state dict: OrderedDict({'linear.weight': tensor([[-0.0031,  0.3723, -0.3986, -0.2375,  0.1004, -0.3590]]), 'linear.bias': tensor([0.1414])})
Optimizer state dict: {'state': {}, 'param_groups': [{'lr': 0.01, 'momentum': 0, 'dampening': 0, 'weight_decay': 0, 'nesterov': False, 'maximize': False, 'foreach': None, 'differentiable': False, 'fused': None, 'params': [0, 1]}]}


## Loading to GPU

In [31]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


### Save to GPU and Load to either CPU or GPU

In [33]:
# save to gpu
PATH = "model-gpu.pth"
model.to(device)
torch.save(model.state_dict(), PATH)

In [35]:
# load on cpu
device = torch.device('cpu')
model = Model(n_input_features=6)
model.load_state_dict(torch.load(PATH, map_location=device))

<All keys matched successfully>

In [36]:
# load on gpu
model = Model(n_input_features=6)
model.load_state_dict(torch.load(PATH))
model.to(device)

Model(
  (linear): Linear(in_features=6, out_features=1, bias=True)
)

### Save to CPU and Load to GPU

In [40]:
# save to cpu
PATH = "model-cpu.pth"
torch.save(model.state_dict(), PATH)

In [41]:
# load on gpu
device = torch.device('cuda')
model = Model(n_input_features=6)
model.load_state_dict(torch.load(PATH, map_location='cuda'))
model.to(device)

Model(
  (linear): Linear(in_features=6, out_features=1, bias=True)
)